# Entity Resolution — Stage 1: Preprocessing Runner (Google Colab)

This notebook provides an automated environment for executing **Stage 1 (Preprocessing)** with separate handling for **Training** and **Testing** datasets on Google Colab.

### Stage 1 Responsibilities:
1. **Unicode NFKD Normalization** (`unicodedata.normalize("NFKD", text)`)
2. **Case Standardization** (Unicode-aware lowercase)
3. **Punctuation & Separator Normalization** (converted to spaces to preserve token boundaries)
4. **Whitespace Normalization**
5. **Non-destructive Tokenization** (stored as JSON arrays)
6. **Preservation of Numbers & Alphanumerics** (stored in `row_numbers` and tokens)
7. **Missing-Value Handling** (no `"nan"` / `"null"` artifacts)
8. **Row Preservation & Schema Validation**

--- 
## 1. Environment Setup & Clone Repository
Switch into the repository directory and install lightweight dependencies.

In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/Blackdevil-com/Wired-Mortus---Amazon-ML.git"
REPO_NAME = "Wired-Mortus---Amazon-ML"

# Ensure we are inside the repository directory
if not Path("src/preprocessing").exists():
    if not Path(REPO_NAME).exists():
        !git clone {REPO_URL}
    %cd {REPO_NAME}

print(f"Current working directory: {Path.cwd()}")
!pip install -r requirements-colab.txt

--- 
## 2. Mount Google Drive & Link Datasets
Mount your Google Drive to load raw training/testing datasets and save preprocessed outputs separately.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Setup Google Drive paths for separate train & test storage
DRIVE_BASE = Path('/content/drive/MyDrive/entity-resolution')
DRIVE_RAW = DRIVE_BASE / 'data/raw'
DRIVE_PROC_TRAIN = DRIVE_BASE / 'data/processed/train'
DRIVE_PROC_TEST = DRIVE_BASE / 'data/processed/test'

# Create local and Drive directories
!mkdir -p data/raw data/processed/train data/processed/test
DRIVE_PROC_TRAIN.mkdir(parents=True, exist_ok=True)
DRIVE_PROC_TEST.mkdir(parents=True, exist_ok=True)

# If raw data exists in Drive, copy to local fast NVMe disk
if DRIVE_RAW.exists():
    print(f"Copying raw datasets from Google Drive ({DRIVE_RAW})...")
    !cp -n "{DRIVE_RAW}"/*.tsv data/raw/ 2>/dev/null || true

print("Local raw dataset files:")
for f in sorted(Path("data/raw").glob("*.tsv")):
    print(f"  - {f.name} ({f.stat().st_size / (1024 * 1024):.2f} MB)")

--- 
## 3. Run Pytest Suite
Verify all Unicode normalization, tokenization, number preservation, and edge cases pass before processing.

In [ ]:
!pytest tests/ -v

--- 
## 4. Preprocess TRAINING Data & Upload to Google Drive
Processes `train_source1.tsv`, `train_source2.tsv`, and `train_source3.tsv` and uploads outputs to `data/processed/train/` in Google Drive.

In [ ]:
# Preprocess Training data
!python run_preprocessing.py --split train --processed-dir data/processed/train

# Copy preprocessed train datasets to Google Drive
print("\nUploading preprocessed train files to Google Drive...")
!cp -v data/processed/train/*.tsv "{DRIVE_PROC_TRAIN}/"

--- 
## 5. Preprocess TESTING Data & Upload to Google Drive
Processes `test_source1.tsv`, `test_source2.tsv`, and `test_source3.tsv` and uploads outputs to `data/processed/test/` in Google Drive.

In [ ]:
# Preprocess Testing data
!python run_preprocessing.py --split test --processed-dir data/processed/test

# Copy preprocessed test datasets to Google Drive
print("\nUploading preprocessed test files to Google Drive...")
!cp -v data/processed/test/*.tsv "{DRIVE_PROC_TEST}/"

--- 
## 6. Inspect Sample Preprocessed Outputs
Display sample records from both train and test preprocessed files to verify normalization, tokens, and numeric preservation.

In [ ]:
import pandas as pd

proc_files = sorted(list(Path("data/processed").glob("**/*_preprocessed.tsv")))
print(f"Preprocessed files available ({len(proc_files)}):\n")

for pfile in proc_files:
    df = pd.read_csv(pfile, sep="\t", nrows=3, dtype=str, keep_default_na=False)
    print("=" * 75)
    print(f"File: {pfile.relative_to('data/processed')} | Records preview:")
    print("=" * 75)
    cols = [c for c in ['entity_id', 'business_name', 'business_name_normalized', 'business_name_tokens', 'business_address_normalized', 'row_numbers'] if c in df.columns]
    display(df[cols])